In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`io.delta::delta-spark_4.1:4.2.0`

import $ivy.$
import $ivy.$

In [2]:
//spark session 
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("ProyectoBigDataGold")
  .master("local[*]")
  .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
  .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 17:34:02 WARN Utils: Your hostname, SamuraiLoco, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/12 17:34:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/12 17:34:02 INFO SparkContext: Running Spark version 4.1.1
26/05/12 17:34:02 INFO SparkContext: OS info Linux, 6.6.114.1-microsoft-standard-WSL2, amd64
26/05/12 17:34:02 INFO SparkContext: Java version 17.0.18+8-Ubuntu-124.04.1
26/05/12 17:34:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/12 17:34:03 INFO ResourceUtils: ==============================================================
26/05/12 17:34:03 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/12 17:34:03 INFO ResourceUtils: ==============================================================
26/05/12 17:34:03

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3cb74816

In [3]:
//rutas 
val basePath = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"

val silverPath = basePath + "/delta/silver/orders_clean"

val goldKpisPath = basePath + "/delta/gold/kpis_generales"
val goldMonthlyPath = basePath + "/delta/gold/ventas_mensuales"
val goldCategoryPath = basePath + "/delta/gold/categorias"
val goldCountryPath = basePath + "/delta/gold/paises"
val goldTopProductsPath = basePath + "/delta/gold/top_productos"

basePath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"
silverPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/silver/orders_clean"
goldKpisPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/kpis_generales"
goldMonthlyPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/ventas_mensuales"
goldCategoryPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/categorias"
goldCountryPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/paises"
goldTopProductsPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/top_productos"

In [4]:
//leemos silver 
val silverDF = spark.read
  .format("delta")
  .load(silverPath)

println(s"Filas Silver: ${silverDF.count()}")
silverDF.printSchema()
silverDF.show(5)

Filas Silver: 10000
root
 |-- category: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- season: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin_pct: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- shipping_method: string (nullable = true)
 |-- shipping_days: int

silverDF: org.apache.spark.sql.package.DataFrame = [category: string, order_id: string ... 29 more fields]

In [5]:
import spark.implicits._

import spark.implicits._

In [6]:
//kpis generales 


val totalOrders = silverDF.select("order_id").distinct().count()

val totalRevenue = silverDF
  .agg(sum("revenue").alias("total_revenue"))
  .first()
  .getAs[Double]("total_revenue")

val totalProfit = silverDF
  .agg(sum("profit").alias("total_profit"))
  .first()
  .getAs[Double]("total_profit")

val avgMargin = silverDF
  .agg(avg("profit_margin_pct").alias("avg_margin"))
  .first()
  .getAs[Double]("avg_margin")

val deliveredOrders = silverDF
  .filter(lower(col("order_status")).contains("delivered"))
  .select("order_id")
  .distinct()
  .count()

val deliveredPct = deliveredOrders.toDouble / totalOrders.toDouble * 100

val kpisGeneralesDF = Seq(
  ("pedidos_totales", totalOrders.toDouble),
  ("ingresos_totales", BigDecimal(totalRevenue).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble),
  ("beneficio_total", BigDecimal(totalProfit).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble),
  ("margen_promedio_pct", BigDecimal(avgMargin / 100).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble),
("porcentaje_pedidos_entregados", BigDecimal(deliveredPct / 100).setScale(2, BigDecimal.RoundingMode.HALF_UP).toDouble)
).toDF("kpi", "valor")

kpisGeneralesDF.show(false)

+-----------------------------+----------+
|kpi                          |valor     |
+-----------------------------+----------+
|pedidos_totales              |10000.0   |
|ingresos_totales             |5284387.7 |
|beneficio_total              |1437638.31|
|margen_promedio_pct          |0.24      |
|porcentaje_pedidos_entregados|0.63      |
+-----------------------------+----------+



totalOrders: Long = 10000L
totalRevenue: Double = 5284387.699999997
totalProfit: Double = 1437638.3099999996
avgMargin: Double = 23.648593999999996
deliveredOrders: Long = 6273L
deliveredPct: Double = 62.73
kpisGeneralesDF: org.apache.spark.sql.package.DataFrame = [kpi: string, valor: double]

In [7]:
// Ventas mensuales
val ventasMensualesDF = silverDF
  .groupBy("year", "month")
  .agg(
    countDistinct("order_id").alias("pedidos"),
    round(sum("revenue"), 2).alias("ingresos"),
    round(sum("profit"), 2).alias("beneficio"),
    round(avg("profit_margin_pct"), 2).alias("margen_promedio_pct")
  )
  .orderBy("year", "month")

ventasMensualesDF.show(20)

+----+-----+-------+---------+---------+-------------------+
|year|month|pedidos| ingresos|beneficio|margen_promedio_pct|
+----+-----+-------+---------+---------+-------------------+
|2021|    1|      7|  9276.43|  3967.15|              26.05|
|2021|    2|     15|  8321.14|  2310.85|              24.91|
|2021|    3|     25|  7377.79|  1914.07|              24.99|
|2021|    4|     44|  27697.6|  4227.08|              19.79|
|2021|    5|     54|  30890.9|  7687.58|              21.56|
|2021|    6|     65|  30822.0|  7524.44|              22.12|
|2021|    7|     68| 32789.34|  9524.55|              24.89|
|2021|    8|    106| 77747.37| 23982.27|              22.11|
|2021|    9|    103| 42899.92| 14508.57|              27.05|
|2021|   10|    115| 76343.18| 23840.24|              26.69|
|2021|   11|    139| 90212.85| 27944.62|              22.65|
|2021|   12|    155| 71681.84| 21099.63|              26.34|
|2022|    1|    151| 99447.76|  26306.7|              22.94|
|2022|    2|    159| 805

ventasMensualesDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [year: int, month: int ... 4 more fields]

In [20]:
// por categorias 
val categoriasDF = silverDF
  .groupBy("category")
  .agg(
      sum("revenue").alias("ingresos"),
      sum("profit").alias("beneficio"),
      sum("quantity").alias("pedidos")
  )

categoriasDF: org.apache.spark.sql.package.DataFrame = [category: string, ingresos: double ... 2 more fields]

In [9]:
//por pais 
val paisesDF = silverDF
  .groupBy("country", "region")
  .agg(
    countDistinct("order_id").alias("pedidos"),
    sum("revenue").alias("ingresos"),
    sum("profit").alias("beneficio"),
    avg("shipping_days").alias("dias_envio_promedio")
  )
  .orderBy(desc("ingresos"))

paisesDF.show(20, false)

+------------+-------------+-------+------------------+------------------+-------------------+
|country     |region       |pedidos|ingresos          |beneficio         |dias_envio_promedio|
+------------+-------------+-------+------------------+------------------+-------------------+
|Canada      |North America|876    |454911.2899999996 |121002.80000000016|6.312785388127854  |
|USA         |North America|829    |444541.87999999995|123170.7200000001 |6.5512665862484925 |
|Mexico      |North America|897    |431676.5799999992 |122543.9000000001 |6.545150501672241  |
|Egypt       |Middle East  |481    |344299.34999999986|103930.98999999993|12.332640332640333 |
|Jordan      |Middle East  |509    |297916.85         |85073.62999999996 |12.620825147347741 |
|China       |Asia         |516    |295780.41000000003|73762.20999999996 |17.424418604651162 |
|Japan       |Asia         |484    |288011.0400000001 |74848.58          |17.359504132231404 |
|Italy       |Europe       |512    |279837.3600000

paisesDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [country: string, region: string ... 4 more fields]

In [16]:
//top productos 
import org.apache.spark.sql.functions._

val topProductosDF = silverDF
  .withColumn("revenue", col("revenue").cast("double"))
  .groupBy("product_name")
  .agg(
    round(sum("revenue"), 2).alias("ingresos"),
    sum("quantity").alias("unidades_vendidas")
  )
  .orderBy(desc("ingresos"))

import org.apache.spark.sql.functions._
topProductosDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product_name: string, ingresos: double ... 1 more field]

In [23]:
//estados de pedido
val estadoPedidosDF = silverDF
  .groupBy("order_status")
  .agg(
    countDistinct("order_id").alias("pedidos")
  )
  .withColumn(
    "porcentaje",
    round(col("pedidos") / sum("pedidos").over() * 100, 2)
  )
  .orderBy(desc("pedidos"))

estadoPedidosDF.show(false)

val goldEstadoPedidosPath = basePath + "/delta/gold/estado_pedidos"


+------------+-------+----------+
|order_status|pedidos|porcentaje|
+------------+-------+----------+
|Delivered   |6273   |62.73     |
|Returned    |1857   |18.57     |
|Processing  |962    |9.62      |
|Cancelled   |908    |9.08      |
+------------+-------+----------+



estadoPedidosDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_status: string, pedidos: bigint ... 1 more field]
goldEstadoPedidosPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/gold/estado_pedidos"

In [24]:
//guardar en delta 
kpisGeneralesDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldKpisPath)

ventasMensualesDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldMonthlyPath)

categoriasDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldCategoryPath)

paisesDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldCountryPath)

topProductosDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldTopProductsPath)

estadoPedidosDF.write
  .format("delta")
  .mode("overwrite")
  .save(goldEstadoPedidosPath)

In [12]:
// y verificamos 
println("Gold guardado correctamente")

println(s"KPIs generales: ${spark.read.format("delta").load(goldKpisPath).count()} filas")
println(s"Ventas mensuales: ${spark.read.format("delta").load(goldMonthlyPath).count()} filas")
println(s"Categorías: ${spark.read.format("delta").load(goldCategoryPath).count()} filas")
println(s"Países: ${spark.read.format("delta").load(goldCountryPath).count()} filas")
println(s"Top productos: ${spark.read.format("delta").load(goldTopProductsPath).count()} filas")

Gold guardado correctamente
KPIs generales: 5 filas
Ventas mensuales: 48 filas
Categorías: 5 filas
Países: 18 filas
Top productos: 10 filas


In [25]:
//ruta para exportar a power Bi 
val exportPath = basePath + "/export_powerbi"

val exportKpis = exportPath + "/kpis_generales"
val exportMensual = exportPath + "/ventas_mensuales"
val exportCategorias = exportPath + "/categorias"
val exportPaises = exportPath + "/paises"
val exportProductos = exportPath + "/top_productos"
val exportEstadoPedidos = exportPath + "/estado_pedidos"

exportPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi"
exportKpis: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/kpis_generales"
exportMensual: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/ventas_mensuales"
exportCategorias: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/categorias"
exportPaises: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/paises"
exportProductos: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/top_productos"
exportEstadoPedidos: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/export_powerbi/estado_pedidos"

In [14]:
//leemos gold desde delta 
val kpisDF = spark.read
  .format("delta")
  .load(goldKpisPath)

val mensualDF = spark.read
  .format("delta")
  .load(goldMonthlyPath)

val categoriasDFExport = spark.read
  .format("delta")
  .load(goldCategoryPath)

val paisesDFExport = spark.read
  .format("delta")
  .load(goldCountryPath)

val productosDFExport = spark.read
  .format("delta")
  .load(goldTopProductsPath)

kpisDF: org.apache.spark.sql.package.DataFrame = [kpi: string, valor: double]
mensualDF: org.apache.spark.sql.package.DataFrame = [year: int, month: int ... 4 more fields]
categoriasDFExport: org.apache.spark.sql.package.DataFrame = [category: string, pedidos: bigint ... 5 more fields]
paisesDFExport: org.apache.spark.sql.package.DataFrame = [country: string, region: string ... 4 more fields]
productosDFExport: org.apache.spark.sql.package.DataFrame = [product_name: string, category: string ... 5 more fields]

In [26]:
//exportamos csv a power bi 
kpisDF.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportKpis)

mensualDF.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportMensual)

categoriasDFExport.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportCategorias)

paisesDFExport.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportPaises)

productosDFExport.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportProductos)

estadoPedidosDF.coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv(exportEstadoPedidos)